# Crop Yield Prediction: End-to-End ML Pipeline
### Objective
Develop a predictive model to estimate crop yield based on historical agricultural data, including factors like crop type, state, area, fertilizer, and pesticide usage.

### Methodology
As a Senior ML Engineer, I will follow these robust ML best practices:
1. **Target Leakage Prevention**: `production` is often mathematically derived from `yield * area`. We must exclude `production` to prevent target leakage.
2. **Exploratory Data Analysis (EDA)**: Understand distributions, outliers, and correlations.
3. **Strict Data Splitting**: Split data into training and test sets *before* applying any transformations to prevent data leakage.
4. **Robust Preprocessing**: Handle missing values, encode categorical variables (One-Hot Encoding for nominal data), and scale numerical features appropriately.
5. **Model Evaluation & Comparison**: Compare a baseline model against a more complex ensemble model (Random Forest) using rigorous metrics (RMSE, R2, MAE).


## 1. Import Libraries
Importing necessary libraries for data manipulation, visualization, and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set plotting style
plt.style.use('ggplot')
sns.set_palette("muted")


## 2. Load and Understand the Data
Let's load the dataset and perform initial checks on its schema, dimensions, and data types.


In [ ]:
# Load dataset
df = pd.read_csv('Dataset/crop_yield.csv')

# Display the first few rows
display(df.head())

# Check data types and non-null counts
print("\n--- Dataset Info ---")
df.info()

# Check for missing values
print("\n--- Missing Values ---")
print(df.isnull().sum())


## 3. Exploratory Data Analysis (EDA)
Understanding the data distribution, finding anomalies, and analyzing relationships between features and the target variable (`yield`).


In [ ]:
# Summary statistics for numerical columns
display(df.describe())


### 3.1 Distribution of the Target Variable (`yield`)


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['yield'], bins=50, kde=True)
plt.title('Distribution of Crop Yield')
plt.xlabel('Yield')
plt.ylabel('Frequency')
plt.show()

# If the distribution is highly skewed, we might consider a log transformation later.


### 3.2 Correlation Analysis
Let's check the correlation between numerical features. We must exclude `production` due to target leakage.


In [ ]:
numerical_cols = ['area', 'fertilizer', 'pesticide', 'yield']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()


### 3.3 Target Leakage Removal
As noted, `production` is heavily correlated with `yield` and `area` (since Production = Area * Yield). Including it as a feature would lead to a severely over-optimistic model (target leakage). We must drop it.


In [ ]:
# Drop target leakage column
if 'production' in df.columns:
    df = df.drop(columns=['production'])
    print("Dropped 'production' to prevent target leakage.")


## 4. Data Preprocessing & Splitting
**CRITICAL ML PRACTICE**: We must split the dataset into training and test sets *before* applying any transformations like scaling or imputation. This ensures that information from the test set does not leak into the training process.


In [ ]:
# Strip whitespace from string columns which is a common data quality issue
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].str.strip()

# Define features and target
X = df.drop(columns=['yield'])
y = df['yield']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")


### 4.1 Build Preprocessing Pipelines
We will use `scikit-learn` Pipelines to handle missing values, scale numerical features, and encode categorical features.


In [ ]:
# Identify categorical and numerical columns
categorical_features = ['crop', 'season', 'state']
# We include 'year' as numerical, though it could be treated as categorical or time-series depending on context.
numerical_features = ['year', 'area', 'fertilizer', 'pesticide']

# Pipeline for numerical features: Impute missing values with median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline for categorical features: Impute with mode, then One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine transformers into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])


## 5. Model Selection and Training
We will compare a baseline model against a regularized linear model and a tree-based ensemble model.
1. **Dummy Regressor**: Always predicts the mean. Acts as our naive baseline.
2. **Ridge Regression**: Linear model with L2 regularization.
3. **Random Forest Regressor**: Non-linear ensemble model that handles complex interactions well.


In [ ]:
# Define models
models = {
    "Baseline (Mean)": DummyRegressor(strategy="mean"),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

# Dictionary to store results
results = {}
trained_pipelines = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"Training {name}...")
    
    # Create a full pipeline combining preprocessor and model
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Fit on training data
    clf.fit(X_train, y_train)
    
    # Predict on test data
    y_pred = clf.predict(X_test)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_pipelines[name] = clf

print("\nTraining complete.")


## 6. Model Evaluation
Let's compare the performance of our models on the test set.


In [ ]:
# Display results as a DataFrame
results_df = pd.DataFrame(results).T
display(results_df)

# Plot R2 scores
plt.figure(figsize=(8, 5))
sns.barplot(x=results_df.index, y=results_df['R2'])
plt.title('Model Comparison: R2 Score (Higher is Better)')
plt.ylabel('R2 Score')
plt.ylim(0, 1)
plt.show()


### 6.1 Actual vs. Predicted (Best Model)
Visualizing how well the Random Forest model predictions align with the actual yield values.


In [ ]:
best_model_name = "Random Forest"
best_model = trained_pipelines[best_model_name]

y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best, alpha=0.3, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Ideal prediction line
plt.title(f'Actual vs Predicted Yield ({best_model_name})')
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')
plt.show()


### 6.2 Feature Importance
Understanding which features drive the predictions in our Random Forest model.


In [ ]:
# Extract the trained Random Forest model and preprocessor
rf_model = best_model.named_steps['model']
preprocessor_step = best_model.named_steps['preprocessor']

# Get feature names after one-hot encoding
cat_features_encoded = preprocessor_step.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_features_encoded)

# Get feature importances
importances = rf_model.feature_importances_

# Create a DataFrame for visualization
importance_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False).head(15) # Top 15 features

# Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.show()


## 7. Conclusion

### Q&A
- **Can we accurately predict crop yield using the provided features?** Yes, by employing a non-linear model like Random Forest, we achieve a significantly higher R2 score compared to a simple linear baseline, demonstrating the model captures complex interactions between crops, regions, and resources.
- **Why was `production` dropped?** Dropping `production` was necessary to prevent target leakage, ensuring our model learns to predict yield based on external factors (area, fertilizer, weather/season, etc.) rather than mathematically deriving it.

### Data Analysis Key Findings
- **Feature Significance**: The feature importance analysis reveals which factors (e.g., specific crops, area, fertilizer amounts) are the strongest predictors of crop yield.
- **Model Performance**: Tree-based ensembles (Random Forest) drastically outperform simple linear models on this dataset, indicating that the relationships between agricultural inputs (fertilizer, pesticide), categorical attributes (season, state, crop), and the output (yield) are highly non-linear.

### Insights or Next Steps
- **Hyperparameter Tuning**: We can further optimize the Random Forest model (e.g., `max_depth`, `min_samples_split`) or try Gradient Boosting (XGBoost/LightGBM) to potentially improve accuracy.
- **Cross-Validation**: Implement k-fold cross-validation particularly with time-series chronological splitting (using `year`) to ensure the model generalizes well to future years.
- **Feature Engineering**: Investigate creating interaction terms (e.g., fertilizer per unit area) to provide more direct signals to the model.
